In [4]:
!pip install xlrd openpyxl


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import glob
import pandas as pd

MASTER_FILE_PATH = r"C:\Users\mahav\Downloads\top20_startups.xlsx"
WIPO_FOLDER_PATH = r"C:\Users\mahav\Downloads\wipo_files"
OUTPUT_FILE_PATH = r"C:\Users\mahav\Downloads\Top_20_Startups_Merged_WIPO_Patents.xlsx"

# 1. Load Master Startup List
startups_df = pd.read_excel(MASTER_FILE_PATH)
startups_df["Clean_Name"] = (
    startups_df["Startup / Company Name"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

# 2. Read WIPO Folder Files
wipo_files = glob.glob(os.path.join(WIPO_FOLDER_PATH, "*.xls*"))
print(f"📁 Found {len(wipo_files)} WIPO files to process.\n")

patent_summary = []

for file_path in wipo_files:
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    clean_file_name = file_name.lower().replace("_", " ").replace("-", " ").strip()
    
    patent_count = 0
    patent_ids = "None / Zero Patents"
    
    try:
        file_ext = os.path.splitext(file_path)[1].lower()
        engine = "xlrd" if file_ext == ".xls" else "openpyxl"
        
        # WIPO exports place headers on row index 3 (4th row)
        wipo_df = pd.read_excel(file_path, header=3, engine=engine)
        
        # Fallback if 'Application Number' isn't in header=3
        if not any('application number' in str(c).lower() for c in wipo_df.columns):
            wipo_df = pd.read_excel(file_path, header=0, engine=engine)
            
        wipo_df = wipo_df.dropna(how="all")
        
        # Target 'Application Number' column explicitly
        target_col = None
        for col in wipo_df.columns:
            if str(col).strip().lower() == 'application number':
                target_col = col
                break
                
        # If not found, look for any column with 'number' or 'id'
        if target_col is None:
            for col in wipo_df.columns:
                if 'number' in str(col).lower() or 'id' in str(col).lower():
                    target_col = col
                    break
        
        if target_col is not None:
            valid_patents = wipo_df[target_col].dropna().astype(str).unique().tolist()
            # Clean out header artifacts or 'nan'
            valid_patents = [p.strip() for p in valid_patents if p.strip().lower() not in ['application number', 'application id', 'nan', 'none', '']]
            patent_count = len(valid_patents)
            if patent_count > 0:
                patent_ids = ", ".join(valid_patents)
        else:
            # Fallback to column index 1 where WIPO application numbers usually reside
            if len(wipo_df.columns) > 1:
                col_data = wipo_df.iloc[:, 1].dropna().astype(str).unique().tolist()
                valid_patents = [p.strip() for p in col_data if p.strip().lower() not in ['application number', 'application id', 'nan', 'none', '']]
                patent_count = len(valid_patents)
                if patent_count > 0:
                    patent_ids = ", ".join(valid_patents)

        print(f"✅ {file_name}: Extracted {patent_count} patent ID(s).")
        
    except Exception as e:
        print(f"⚠️ Error reading {file_name}: {e}")

    patent_summary.append({
        "Clean_Name": clean_file_name,
        "Total_Patents_WIPO": patent_count,
        "Patent_Application_IDs": patent_ids
    })

patent_summary_df = pd.DataFrame(patent_summary)

# 3. Merge with Master List
merged_df = pd.merge(startups_df, patent_summary_df, on="Clean_Name", how="left")
merged_df["Total_Patents_WIPO"] = merged_df["Total_Patents_WIPO"].fillna(0).astype(int)
merged_df["Patent_Application_IDs"] = merged_df["Patent_Application_IDs"].fillna("None / Zero Patents")

final_df = merged_df.drop(columns=["Clean_Name"], errors="ignore")

# 4. Save Final Output
final_df.to_excel(OUTPUT_FILE_PATH, index=False)
print(f"\n🎉 Successfully saved updated spreadsheet with exact Patent IDs to:\n{OUTPUT_FILE_PATH}")

📁 Found 18 WIPO files to process.

✅ Agnikul Cosmos: Extracted 49 patent ID(s).
✅ Ather Energy: Extracted 103 patent ID(s).
✅ Attero Recycling: Extracted 75 patent ID(s).
✅ Battery Smart: Extracted 16 patent ID(s).
✅ Chakr Innovation: Extracted 16 patent ID(s).
✅ Climes: Extracted 7 patent ID(s).
✅ Ecozen Solutions: Extracted 4 patent ID(s).
✅ Exponent Energy: Extracted 4 patent ID(s).
✅ GreenPod Labs: Extracted 8 patent ID(s).
✅ Kazam: Extracted 4 patent ID(s).
✅ Lohum Cleantech: Extracted 8 patent ID(s).
✅ Ola Electric: Extracted 273 patent ID(s).
✅ Phool.co: Extracted 6 patent ID(s).
✅ Praan: Extracted 8 patent ID(s).
✅ Renew: Extracted 14 patent ID(s).
✅ String Bio: Extracted 50 patent ID(s).
✅ Uravu Labs: Extracted 16 patent ID(s).
✅ Varaha: Extracted 6 patent ID(s).

🎉 Successfully saved updated spreadsheet with exact Patent IDs to:
C:\Users\mahav\Downloads\Top_20_Startups_Merged_WIPO_Patents.xlsx
